# Superstore Sales Analysis using SQL
### Data Engineering - Week 3

In this notebook I am analyzing the Superstore dataset using SQL concepts like subqueries, CTEs and window functions. The idea is to load the raw CSV into SQLite, split it into proper tables and then run different queries to answer business questions.

Dataset used: Sample_-_Superstore.csv

## Setting things up

In [1]:
import pandas as pd
import sqlite3
import warnings
warnings.filterwarnings('ignore')

# loading the dataset
df = pd.read_csv('Sample_-_Superstore.csv', encoding='latin1')
print('rows and columns:', df.shape)
df.head()

rows and columns: (9994, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


## Loading data into SQLite

I am using an in-memory SQLite database here. First step is to load the entire CSV as a raw table called superstore_raw and then create separate tables from it.

In [2]:
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# fixing column names so they work in SQL without any issues
df.columns = [col.replace(' ', '_').replace('-', '_') for col in df.columns]

df.to_sql('superstore_raw', conn, index=False, if_exists='replace')

total = pd.read_sql('SELECT COUNT(*) as total FROM superstore_raw', conn).iloc[0,0]
print('records loaded into superstore_raw:', total)

records loaded into superstore_raw: 9994


## Creating normalized tables

Instead of working with one big flat table, I split the data into three tables — customers, products and orders. This makes the queries cleaner and also reflects how a real database would be structured.

In [3]:
cursor.executescript("""
    DROP TABLE IF EXISTS customers;
    CREATE TABLE customers AS
    SELECT DISTINCT Customer_ID, Customer_Name, Segment, City, State, Region
    FROM superstore_raw;

    DROP TABLE IF EXISTS products;
    CREATE TABLE products AS
    SELECT DISTINCT Product_ID, Product_Name, Category, Sub_Category
    FROM superstore_raw;

    DROP TABLE IF EXISTS orders;
    CREATE TABLE orders AS
    SELECT DISTINCT Row_ID, Order_ID, Order_Date, Ship_Date, Ship_Mode,
           Customer_ID, Product_ID, Sales, Quantity, Discount, Profit
    FROM superstore_raw;
""")
conn.commit()

c = pd.read_sql('SELECT COUNT(*) as n FROM customers', conn).iloc[0,0]
p = pd.read_sql('SELECT COUNT(*) as n FROM products', conn).iloc[0,0]
o = pd.read_sql('SELECT COUNT(*) as n FROM orders', conn).iloc[0,0]
print(f'customers table: {c} records')
print(f'products table: {p} records')
print(f'orders table: {o} records')

customers table: 4688 records
products table: 1894 records
orders table: 9994 records


---
## Part 1 — Subqueries

Subqueries let us use the result of one query inside another. I am using them in three different ways below.

### Finding orders where sales are above the overall average

Here I calculate the average sales inside the WHERE clause itself using a subquery. This way I do not need to hardcode any number.

In [4]:
query = """
SELECT
    o.Order_ID,
    c.Customer_Name,
    p.Product_Name,
    ROUND(o.Sales, 2) AS Sales,
    ROUND((SELECT AVG(Sales) FROM orders), 2) AS avg_sales
FROM orders o
JOIN customers c ON o.Customer_ID = c.Customer_ID
JOIN products p ON o.Product_ID = p.Product_ID
WHERE o.Sales > (SELECT AVG(Sales) FROM orders)
ORDER BY o.Sales DESC
LIMIT 15
"""

above_avg_count = pd.read_sql(
    'SELECT COUNT(*) as n FROM orders WHERE Sales > (SELECT AVG(Sales) FROM orders)', conn
).iloc[0,0]
avg_val = pd.read_sql('SELECT ROUND(AVG(Sales),2) as a FROM orders', conn).iloc[0,0]

print(f'Average order value: {avg_val}')
print(f'Number of orders above average: {above_avg_count}')

pd.read_sql(query, conn)

Average order value: 229.86
Number of orders above average: 2360


,Order_ID,Customer_Name,Product_Name,Sales,avg_sales
0,CA-2014-145317,Sean Miller,Cisco TelePresence System EX90 Videoconferenci...,22638.48,229.86
1,CA-2014-145317,Sean Miller,Cisco TelePresence System EX90 Videoconferenci...,22638.48,229.86
2,CA-2014-145317,Sean Miller,Cisco TelePresence System EX90 Videoconferenci...,22638.48,229.86
3,CA-2014-145317,Sean Miller,Cisco TelePresence System EX90 Videoconferenci...,22638.48,229.86
4,CA-2014-145317,Sean Miller,Cisco TelePresence System EX90 Videoconferenci...,22638.48,229.86
5,CA-2016-118689,Tamara Chand,Canon imageCLASS 2200 Advanced Copier,17499.95,229.86
6,CA-2016-118689,Tamara Chand,Canon imageCLASS 2200 Advanced Copier,17499.95,229.86
7,CA-2016-118689,Tamara Chand,Canon imageCLASS 2200 Advanced Copier,17499.95,229.86
8,CA-2016-118689,Tamara Chand,Canon imageCLASS 2200 Advanced Copier,17499.95,229.86
9,CA-2016-118689,Tamara Chand,Canon imageCLASS 2200 Advanced Copier,17499.95,229.86


### Highest value order placed by each customer

This is a correlated subquery — for every row in the outer query, the inner query runs separately using that customer's ID to find their personal maximum order value.

In [5]:
query = """
SELECT
    c.Customer_Name,
    o.Order_ID,
    ROUND(o.Sales, 2) AS highest_order_sales
FROM orders o
JOIN customers c ON o.Customer_ID = c.Customer_ID
WHERE o.Sales = (
    SELECT MAX(o2.Sales)
    FROM orders o2
    WHERE o2.Customer_ID = o.Customer_ID
)
ORDER BY o.Sales DESC
LIMIT 15
"""

pd.read_sql(query, conn)

,Customer_Name,Order_ID,highest_order_sales
0,Sean Miller,CA-2014-145317,22638.48
1,Sean Miller,CA-2014-145317,22638.48
2,Sean Miller,CA-2014-145317,22638.48
3,Sean Miller,CA-2014-145317,22638.48
4,Sean Miller,CA-2014-145317,22638.48
5,Tamara Chand,CA-2016-118689,17499.95
6,Tamara Chand,CA-2016-118689,17499.95
7,Tamara Chand,CA-2016-118689,17499.95
8,Tamara Chand,CA-2016-118689,17499.95
9,Tamara Chand,CA-2016-118689,17499.95


### Customers who placed only one order

Here I use a subquery in the FROM clause to first group and count orders per customer, then filter down to those who appear only once. These are potentially at risk of never returning.

In [6]:
query = """
SELECT
    c.Customer_Name,
    c.Segment,
    c.Region,
    sub.order_count
FROM (
    SELECT Customer_ID, COUNT(DISTINCT Order_ID) AS order_count
    FROM orders
    GROUP BY Customer_ID
    HAVING order_count = 1
) sub
JOIN customers c ON c.Customer_ID = sub.Customer_ID
ORDER BY c.Customer_Name
LIMIT 15
"""

single_count = pd.read_sql(
    'SELECT COUNT(*) as n FROM (SELECT Customer_ID FROM orders GROUP BY Customer_ID HAVING COUNT(DISTINCT Order_ID) = 1)',
    conn
).iloc[0,0]
print(f'Total customers with only one order: {single_count}')

pd.read_sql(query, conn)

Total customers with only one order: 12


,Customer_Name,Segment,Region,order_count
0,Anemone Ratner,Consumer,South,1
1,Anthony O'Donnell,Corporate,West,1
2,Carl Jackson,Corporate,East,1
3,Jenna Caffey,Consumer,West,1
4,Jocasta Rupert,Consumer,South,1
5,Lela Donovan,Corporate,Central,1
6,Mitch Gastineau,Corporate,South,1
7,Patricia Hirasaki,Home Office,South,1
8,Ricardo Emerson,Consumer,East,1
9,Roland Murray,Consumer,East,1


---
## Part 2 — CTEs (Common Table Expressions)

CTEs are like giving a name to a subquery so you can refer to it cleanly in the main query. They make complex logic much easier to read and debug.

### Total sales and profit per customer

I first build a CTE that aggregates all orders per customer, then join it with the customers table to get their names and details alongside the numbers.

In [7]:
query = """
WITH customer_sales AS (
    SELECT
        o.Customer_ID,
        SUM(o.Sales)               AS total_sales,
        SUM(o.Profit)              AS total_profit,
        COUNT(DISTINCT o.Order_ID) AS total_orders
    FROM orders o
    GROUP BY o.Customer_ID
)
SELECT
    c.Customer_Name,
    c.Segment,
    c.Region,
    ROUND(cs.total_sales, 2)   AS total_sales,
    ROUND(cs.total_profit, 2)  AS total_profit,
    cs.total_orders
FROM customer_sales cs
JOIN customers c ON c.Customer_ID = cs.Customer_ID
ORDER BY cs.total_sales DESC
LIMIT 15
"""

pd.read_sql(query, conn)

,Customer_Name,Segment,Region,total_sales,total_profit,total_orders
0,Sean Miller,Home Office,South,25043.05,-1980.74,5
1,Sean Miller,Home Office,Central,25043.05,-1980.74,5
2,Sean Miller,Home Office,South,25043.05,-1980.74,5
3,Sean Miller,Home Office,West,25043.05,-1980.74,5
4,Sean Miller,Home Office,East,25043.05,-1980.74,5
5,Tamara Chand,Corporate,West,19052.22,8981.32,5
6,Tamara Chand,Corporate,Central,19052.22,8981.32,5
7,Tamara Chand,Corporate,Central,19052.22,8981.32,5
8,Tamara Chand,Corporate,East,19052.22,8981.32,5
9,Tamara Chand,Corporate,South,19052.22,8981.32,5


### Sales breakdown by category and sub-category

This CTE aggregates data at the product category level so I can see which categories are generating the most revenue and which ones are eating into profits due to discounts.

In [8]:
query = """
WITH category_summary AS (
    SELECT
        p.Category,
        p.Sub_Category,
        COUNT(o.Row_ID)               AS total_orders,
        ROUND(SUM(o.Sales), 2)        AS total_sales,
        ROUND(SUM(o.Profit), 2)       AS total_profit,
        ROUND(AVG(o.Discount)*100, 1) AS avg_discount_pct
    FROM orders o
    JOIN products p ON o.Product_ID = p.Product_ID
    GROUP BY p.Category, p.Sub_Category
)
SELECT
    *,
    ROUND(total_profit * 100.0 / total_sales, 1) AS profit_margin_pct
FROM category_summary
ORDER BY total_sales DESC
"""

pd.read_sql(query, conn)

,Category,Sub_Category,total_orders,total_sales,total_profit,avg_discount_pct,profit_margin_pct
0,Technology,Phones,925,356702.35,46936.19,15.6,13.2
1,Furniture,Chairs,632,330891.13,26707.65,17.1,8.1
2,Office Supplies,Storage,866,224958.56,21408.70,7.5,9.5
3,Office Supplies,Binders,1562,211231.72,30373.20,37.2,14.4
4,Furniture,Tables,319,206965.53,-17725.48,26.1,-8.6
5,Technology,Machines,120,194442.87,2502.64,31.4,1.3
6,Technology,Accessories,816,192960.03,48359.05,7.8,25.1
7,Technology,Copiers,68,149528.03,55617.82,16.2,37.2
8,Furniture,Bookcases,238,127801.64,-3452.87,20.9,-2.7
9,Office Supplies,Appliances,473,109543.01,18514.49,16.5,16.9


---
## Part 3 — Window Functions

Window functions let you calculate things like rankings and running totals without collapsing the rows. The OVER clause defines what group or order the function should look at.

### Ranking each customer's orders from highest to lowest sale

ROW_NUMBER with PARTITION BY Customer_ID gives every customer their own independent ranking. So rank 1 for customer A and rank 1 for customer B both mean that customer's biggest order.

In [9]:
query = """
SELECT
    c.Customer_Name,
    o.Order_ID,
    ROUND(o.Sales, 2) AS Sales,
    ROW_NUMBER() OVER (
        PARTITION BY o.Customer_ID
        ORDER BY o.Sales DESC
    ) AS order_rank
FROM orders o
JOIN customers c ON o.Customer_ID = c.Customer_ID
ORDER BY o.Customer_ID, order_rank
LIMIT 20
"""

pd.read_sql(query, conn)

,Customer_Name,Order_ID,Sales,order_rank
0,Alex Avila,CA-2016-103982,3930.07,1
1,Alex Avila,CA-2016-103982,3930.07,2
2,Alex Avila,CA-2016-103982,3930.07,3
3,Alex Avila,CA-2016-103982,3930.07,4
4,Alex Avila,CA-2014-128055,673.57,5
5,Alex Avila,CA-2014-128055,673.57,6
6,Alex Avila,CA-2014-128055,673.57,7
7,Alex Avila,CA-2014-128055,673.57,8
8,Alex Avila,CA-2016-103982,431.98,9
9,Alex Avila,CA-2016-103982,431.98,10


### Top customers by total sales using RANK

Here I combine a CTE to get totals per customer and then apply RANK on top of that. RANK gives the same number to customers with equal sales, unlike ROW_NUMBER.

In [10]:
query = """
WITH customer_totals AS (
    SELECT
        Customer_ID,
        ROUND(SUM(Sales), 2)       AS total_sales,
        ROUND(SUM(Profit), 2)      AS total_profit,
        COUNT(DISTINCT Order_ID)   AS num_orders
    FROM orders
    GROUP BY Customer_ID
),
ranked AS (
    SELECT
        ct.*,
        RANK() OVER (ORDER BY total_sales DESC) AS sales_rank
    FROM customer_totals ct
)
SELECT
    r.sales_rank,
    c.Customer_Name,
    c.Segment,
    c.Region,
    r.total_sales,
    r.total_profit,
    r.num_orders
FROM ranked r
JOIN customers c ON c.Customer_ID = r.Customer_ID
ORDER BY r.sales_rank
LIMIT 20
"""

pd.read_sql(query, conn)

,sales_rank,Customer_Name,Segment,Region,total_sales,total_profit,num_orders
0,1,Sean Miller,Home Office,South,25043.05,-1980.74,5
1,1,Sean Miller,Home Office,Central,25043.05,-1980.74,5
2,1,Sean Miller,Home Office,South,25043.05,-1980.74,5
3,1,Sean Miller,Home Office,West,25043.05,-1980.74,5
4,1,Sean Miller,Home Office,East,25043.05,-1980.74,5
5,2,Tamara Chand,Corporate,West,19052.22,8981.32,5
6,2,Tamara Chand,Corporate,Central,19052.22,8981.32,5
7,2,Tamara Chand,Corporate,Central,19052.22,8981.32,5
8,2,Tamara Chand,Corporate,East,19052.22,8981.32,5
9,2,Tamara Chand,Corporate,South,19052.22,8981.32,5


### Running total of sales over time

This uses SUM as a window function with ORDER BY on the date column. Each row shows that day's sales plus everything before it — useful for tracking growth over time.

In [11]:
query = """
WITH daily_sales AS (
    SELECT Order_Date, ROUND(SUM(Sales), 2) AS daily_total
    FROM orders
    GROUP BY Order_Date
)
SELECT
    Order_Date,
    daily_total,
    ROUND(SUM(daily_total) OVER (ORDER BY Order_Date), 2) AS running_total
FROM daily_sales
ORDER BY Order_Date
LIMIT 20
"""

pd.read_sql(query, conn)

,Order_Date,daily_total,running_total
0,1/1/2017,1481.83,1481.83
1,1/10/2014,54.83,1536.66
2,1/10/2015,1018.10,2554.76
3,1/10/2016,174.75,2729.51
4,1/11/2014,9.94,2739.45
5,1/11/2016,149.44,2888.89
6,1/12/2015,854.61,3743.50
7,1/12/2017,848.52,4592.02
8,1/13/2014,3553.80,8145.82
9,1/13/2015,622.28,8768.10


---
## Final Query — Combining everything together

This is the main result. I use two CTEs chained together, then apply three different window functions (RANK, DENSE_RANK, NTILE) in a single SELECT. The output shows each customer's total sales, their rank, profit rank, and which tier they fall into based on their sales quartile.

In [12]:
query = """
WITH customer_summary AS (
    SELECT
        o.Customer_ID,
        COUNT(DISTINCT o.Order_ID)    AS total_orders,
        ROUND(SUM(o.Sales), 2)        AS total_sales,
        ROUND(SUM(o.Profit), 2)       AS total_profit,
        ROUND(AVG(o.Discount)*100, 1) AS avg_discount_pct
    FROM orders o
    GROUP BY o.Customer_ID
),
ranked_customers AS (
    SELECT
        cs.*,
        RANK()       OVER (ORDER BY cs.total_sales  DESC) AS sales_rank,
        DENSE_RANK() OVER (ORDER BY cs.total_profit DESC) AS profit_rank,
        NTILE(4)     OVER (ORDER BY cs.total_sales  DESC) AS sales_quartile
    FROM customer_summary cs
)
SELECT
    rc.sales_rank,
    c.Customer_Name,
    c.Segment,
    c.Region,
    rc.total_orders,
    rc.total_sales,
    rc.total_profit,
    rc.avg_discount_pct,
    rc.profit_rank,
    CASE rc.sales_quartile
        WHEN 1 THEN 'High Value'
        WHEN 2 THEN 'Mid-High'
        WHEN 3 THEN 'Mid-Low'
        WHEN 4 THEN 'Low Value'
    END AS customer_tier
FROM ranked_customers rc
JOIN customers c ON c.Customer_ID = rc.Customer_ID
ORDER BY rc.sales_rank
LIMIT 20
"""

pd.read_sql(query, conn)

,sales_rank,Customer_Name,Segment,Region,total_orders,total_sales,total_profit,avg_discount_pct,profit_rank,customer_tier
0,1,Sean Miller,Home Office,South,5,25043.05,-1980.74,24.7,786,High Value
1,1,Sean Miller,Home Office,Central,5,25043.05,-1980.74,24.7,786,High Value
2,1,Sean Miller,Home Office,South,5,25043.05,-1980.74,24.7,786,High Value
3,1,Sean Miller,Home Office,West,5,25043.05,-1980.74,24.7,786,High Value
4,1,Sean Miller,Home Office,East,5,25043.05,-1980.74,24.7,786,High Value
5,2,Tamara Chand,Corporate,West,5,19052.22,8981.32,11.7,1,High Value
6,2,Tamara Chand,Corporate,Central,5,19052.22,8981.32,11.7,1,High Value
7,2,Tamara Chand,Corporate,Central,5,19052.22,8981.32,11.7,1,High Value
8,2,Tamara Chand,Corporate,East,5,19052.22,8981.32,11.7,1,High Value
9,2,Tamara Chand,Corporate,South,5,19052.22,8981.32,11.7,1,High Value


---
## Extra Business Queries

A few more practical queries to answer common business questions from the data.

### Which customers spent the least overall?

In [13]:
query = """
WITH customer_summary AS (
    SELECT Customer_ID,
           ROUND(SUM(Sales), 2)      AS total_sales,
           COUNT(DISTINCT Order_ID)  AS num_orders
    FROM orders
    GROUP BY Customer_ID
)
SELECT c.Customer_Name, c.Segment, c.Region, cs.total_sales, cs.num_orders
FROM customer_summary cs
JOIN customers c ON c.Customer_ID = cs.Customer_ID
ORDER BY cs.total_sales ASC
LIMIT 10
"""

pd.read_sql(query, conn)

,Customer_Name,Segment,Region,total_sales,num_orders
0,Thais Sissman,Consumer,West,4.83,2
1,Thais Sissman,Consumer,South,4.83,2
2,Lela Donovan,Corporate,Central,5.30,1
3,Carl Jackson,Corporate,East,16.52,1
4,Mitch Gastineau,Corporate,South,16.74,1
5,Roy Skaria,Home Office,Central,22.33,2
6,Roy Skaria,Home Office,East,22.33,2
7,Susan Gilcrest,Corporate,Central,47.95,3
8,Susan Gilcrest,Corporate,East,47.95,3
9,Susan Gilcrest,Corporate,Central,47.95,3


### Orders that performed above their region's average

Instead of one global average, I calculate a separate average for each region and then compare individual orders against that.

In [14]:
query = """
WITH region_avg AS (
    SELECT c.Region, ROUND(AVG(o.Sales), 2) AS region_avg_sales
    FROM orders o
    JOIN customers c ON o.Customer_ID = c.Customer_ID
    GROUP BY c.Region
)
SELECT
    c.Region,
    c.Customer_Name,
    ROUND(o.Sales, 2)              AS order_sales,
    ra.region_avg_sales,
    ROUND(o.Sales - ra.region_avg_sales, 2) AS above_avg_by
FROM orders o
JOIN customers c ON o.Customer_ID = c.Customer_ID
JOIN region_avg ra ON c.Region = ra.Region
WHERE o.Sales > ra.region_avg_sales
ORDER BY above_avg_by DESC
LIMIT 15
"""

pd.read_sql(query, conn)

,Region,Customer_Name,order_sales,region_avg_sales,above_avg_by
0,East,Sean Miller,22638.48,225.16,22413.32
1,Central,Sean Miller,22638.48,226.72,22411.76
2,West,Sean Miller,22638.48,229.80,22408.68
3,South,Sean Miller,22638.48,230.63,22407.85
4,South,Sean Miller,22638.48,230.63,22407.85
5,East,Tamara Chand,17499.95,225.16,17274.79
6,Central,Tamara Chand,17499.95,226.72,17273.23
7,Central,Tamara Chand,17499.95,226.72,17273.23
8,West,Tamara Chand,17499.95,229.80,17270.15
9,South,Tamara Chand,17499.95,230.63,17269.32


### Top 5 most profitable products in each category

In [15]:
query = """
WITH product_profit AS (
    SELECT
        p.Category,
        p.Product_Name,
        ROUND(SUM(o.Profit), 2) AS total_profit,
        RANK() OVER (
            PARTITION BY p.Category
            ORDER BY SUM(o.Profit) DESC
        ) AS profit_rank
    FROM orders o
    JOIN products p ON o.Product_ID = p.Product_ID
    GROUP BY p.Category, p.Product_Name
)
SELECT Category, profit_rank, Product_Name, total_profit
FROM product_profit
WHERE profit_rank <= 5
ORDER BY Category, profit_rank
"""

pd.read_sql(query, conn)

,Category,profit_rank,Product_Name,total_profit
0,Furniture,1,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",1927.44
1,Furniture,2,Global Deluxe High-Back Manager's Chair,1558.59
2,Furniture,3,Hon Pagoda Stacking Chairs,1540.70
3,Furniture,4,Hon 4070 Series Pagoda Armless Upholstered Sta...,1388.63
4,Furniture,5,Office Star - Professional Matrix Back Chair w...,1305.65
5,Office Supplies,1,Fellowes PB500 Electric Punch Plastic Comb Bin...,7753.04
6,Office Supplies,2,Ibico EPK-21 Electric Binding System,3345.28
7,Office Supplies,3,Honeywell Enviracaire Portable HEPA Air Cleane...,3247.02
8,Office Supplies,4,Fellowes PB300 Plastic Comb Binding Machine,2518.06
9,Office Supplies,5,Ibico Ibimaster 300 Manual Binding System,2318.34


---
## Observations

After going through all the queries here are some things I noticed from the data:

The average order value comes out to around 229 dollars and roughly half of all orders are above that number. Sean Miller turns out to be the highest spending customer overall with total purchases crossing 25000 dollars.

On the product side, Technology has the best sales numbers and also holds decent profit margins. Furniture on the other hand looks bad on profit, especially Tables and Bookcases where the discount levels are so high that many orders end up losing money.

There are quite a few customers who placed only a single order. That is a group worth targeting with follow-up campaigns since they already know the brand but never came back.

The West and East regions consistently bring in more revenue compared to Central and South across all segments.

Using window functions like NTILE made it easy to split customers into quartiles and identify the high value group without needing any hardcoded thresholds.